## 모델 학습 진행


데이터셋 shape를 직접 확인해드리겠습니다.**결과 평가:**

- AUC 0.848은 "이 종목 자신의 평소 수준과 비교"라는 더 어렵고 현실적인 문제에서 0.84 이상이면 강한 신호력입니다 (참고로 횡단면 방식은 0.75였습니다).
- F1 0.758은 target 비율이 38~43%로 어느 정도 균형 잡혀 있어 신뢰할 만한 수치입니다.
- LightGBM이 LR보다 AUC +0.029, F1 +0.032로 일관되게 우위 — "비선형 모델의 우위"라는 발표 메시지에 정확히 부합합니다.
- Train(0.377) vs Test(0.428) target 비율 차이는 자연스러운 수준(시장 전체 공매도 활동이 후반기에 다소 증가했을 가능성)이고, 모델이 이 변화에도 잘 적응한 것으로 보입니다.

**데이터셋 shape:**

| 단계 | shape | 설명 |
|---|---|---|
| 원본 병합 (load_data) | (248,125, 15) | price + short_merged, 948개 종목 × 263거래일 (2025-03-04~2026-03-31) |
| 피처/타깃 추가 후 | (248,125, 47) | 26개 피처 + 타깃 관련 컬럼 |
| Train (dropna 후) | (152,816, 47) | 901종목 × 193일 |
| Val | (15,971, 47) | 887종목 × 20일 |
| Test | (16,489, 47) | 907종목 × 20일 |
| 학습 가능 행 합계 | 185,276 / 248,125 (74.7%) | rolling/expanding window로 초기 구간(MIN_HIST_DAYS=30일, 20일 이동평균 등) 소실 |


In [2]:
"""
short_model.py  —  모델 학습 전용 모듈
XAI(SHAP/DiCE) 분석은 short_xai.py 참고

포함 내용:
  - 데이터 로드 & 전처리
  - 피처 엔지니어링
  - 타깃 정의
  - 시계열 분할
  - 모델 학습 (Linear/Logistic/LightGBM/XGBoost/CatBoost/LSTM)
  - 학습 결과 저장 (models.joblib, df_full.pkl)
  - 종목 조회 유틸 (predict_for, predict_history)
"""

import time
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, f1_score
import lightgbm as lgb
import xgboost as xgb
import catboost as cb

# ------------------------------------------------------------------
# 상수
# ------------------------------------------------------------------
RANDOM_STATE  = 42
HORIZON_DAYS  = 5
VAL_SIZE_DAYS = 20
TEST_SIZE_DAYS = 20
Q             = 0.7
MIN_HIST_DAYS = 30
LOOKBACK      = 10
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"

FEATURE_COLS = [
    "log_mktcap", "log_price", "is_large_cap",
    "ret_1d", "ret_5d", "ret_20d", "vol_5d", "vol_20d", "intraday_vol", "price_ma20_gap",
    "turnover", "log_trdval", "abnormal_vol", "trdval_ma5_over_ma20", "ret5_x_abnormal_vol",
    "short_ratio_ma5", "short_ratio_ma10", "short_ratio_lag1",
    "short_ratio_chg_1d", "short_ratio_std5",
    "balance_ratio_chg_5d", "balance_ratio_ma5", "balance_ratio_lag1", "short_qty_chg5",
    "short_ratio_pct_vs_own_hist", "balance_ratio_pct_vs_own_hist",
]


# ------------------------------------------------------------------
# 공통 유틸
# ------------------------------------------------------------------
def find_best_threshold(y_val, proba_val, lo=0.2, hi=0.8, step=0.01):
    """val F1을 최대화하는 임계값 반환."""
    best_thresh, best_f1 = 0.5, -1.0
    for t in np.arange(lo, hi, step):
        f = f1_score(y_val, proba_val >= t, zero_division=0)
        if f > best_f1:
            best_f1, best_thresh = f, t
    return round(float(best_thresh), 3)


# ------------------------------------------------------------------
# 1. 데이터 로드 & 병합
# ------------------------------------------------------------------
def clean_numeric(s):
    return pd.to_numeric(s.astype(str).str.replace(",", ""), errors="coerce")


def load_data():
    price  = pd.read_csv("./dataset/df_price_20250401_20260331_with_60d_buffer.csv")
    merged = pd.read_csv("./dataset/df_short_merged_20250304_20260331.csv")

    for d in (price, merged):
        d["ISU_CD"] = d["ISU_CD"].astype(str).str.zfill(6)
        d["date"]   = pd.to_datetime(d["date"])

    num_cols_price = ["TDD_OPNPRC", "TDD_HGPRC", "TDD_LWPRC", "TDD_CLSPRC",
                      "ACC_TRDVOL", "CHG_RT"]
    num_cols_short = ["short_qty", "acc_trdvol", "short_ratio",
                      "balance_ratio", "MKTCAP", "LIST_SHRS"]
    price[num_cols_price]   = price[num_cols_price].apply(clean_numeric)
    merged[num_cols_short]  = merged[num_cols_short].apply(clean_numeric)

    df = price.merge(
        merged[["ISU_CD", "date", "short_qty", "short_ratio",
                "balance_ratio", "MKTCAP", "LIST_SHRS", "acc_trdvol"]],
        on=["ISU_CD", "date"], how="inner",
    )
    return df.sort_values(["ISU_CD", "date"]).reset_index(drop=True)


# ------------------------------------------------------------------
# 2. 피처 엔지니어링
# ------------------------------------------------------------------
def add_features(df):
    g = df.groupby("ISU_CD")

    df["log_mktcap"]   = np.log(df["MKTCAP"].clip(lower=1))
    df["log_price"]    = np.log(df["TDD_CLSPRC"].clip(lower=1))
    df["is_large_cap"] = (df["MKTCAP"] >= g["MKTCAP"].transform("median")).astype(int)

    df["ret_1d"]  = g["TDD_CLSPRC"].pct_change(1)
    df["ret_5d"]  = g["TDD_CLSPRC"].pct_change(5)
    df["ret_20d"] = g["TDD_CLSPRC"].pct_change(20)
    df["vol_5d"]  = g["ret_1d"].transform(lambda x: x.rolling(5).std())
    df["vol_20d"] = g["ret_1d"].transform(lambda x: x.rolling(20).std())
    df["intraday_vol"]   = (df["TDD_HGPRC"] - df["TDD_LWPRC"]) / df["TDD_CLSPRC"]
    ma20 = g["TDD_CLSPRC"].transform(lambda x: x.rolling(20).mean())
    df["price_ma20_gap"] = (df["TDD_CLSPRC"] - ma20) / ma20

    df["turnover"]   = df["ACC_TRDVOL"] / df["LIST_SHRS"]
    trdval = df["ACC_TRDVOL"] * df["TDD_CLSPRC"]
    df["log_trdval"] = np.log(trdval.clip(lower=1))
    vol_avg20 = g["ACC_TRDVOL"].transform(lambda x: x.rolling(20).mean())
    df["abnormal_vol"] = df["ACC_TRDVOL"] / vol_avg20
    trdval_ma5  = trdval.groupby(df["ISU_CD"]).transform(lambda x: x.rolling(5).mean())
    trdval_ma20 = trdval.groupby(df["ISU_CD"]).transform(lambda x: x.rolling(20).mean())
    df["trdval_ma5_over_ma20"] = trdval_ma5 / trdval_ma20
    df["ret5_x_abnormal_vol"]  = df["ret_5d"] * df["abnormal_vol"]

    df["short_ratio_ma5"]      = g["short_ratio"].transform(lambda x: x.rolling(5).mean())
    df["short_ratio_ma10"]     = g["short_ratio"].transform(lambda x: x.rolling(10).mean())
    df["short_ratio_lag1"]     = g["short_ratio"].shift(1)
    df["short_ratio_chg_1d"]   = g["short_ratio"].diff(1)
    df["short_ratio_std5"]     = g["short_ratio"].transform(lambda x: x.rolling(5).std())
    df["balance_ratio_chg_5d"] = g["balance_ratio"].diff(5)
    df["balance_ratio_ma5"]    = g["balance_ratio"].transform(lambda x: x.rolling(5).mean())
    df["balance_ratio_lag1"]   = g["balance_ratio"].shift(1)
    df["short_qty_chg5"]       = g["short_qty"].pct_change(5).replace([np.inf, -np.inf], np.nan)

    df["short_ratio_pct_vs_own_hist"] = g["short_ratio"].transform(
        lambda x: x.shift(1).expanding(min_periods=MIN_HIST_DAYS).rank(pct=True)
    )
    df["balance_ratio_pct_vs_own_hist"] = g["balance_ratio"].transform(
        lambda x: x.shift(1).expanding(min_periods=MIN_HIST_DAYS).rank(pct=True)
    )
    return df


# ------------------------------------------------------------------
# 3. 타깃 정의
# ------------------------------------------------------------------
def add_target(df):
    g = df.groupby("ISU_CD")

    df["future_short_ratio_median_h5"] = (
        g["short_ratio"].transform(lambda x: x.shift(-1).rolling(HORIZON_DAYS).median())
    )
    df["future_balance_ratio_h5"] = g["balance_ratio"].shift(-HORIZON_DAYS)
    df["future_ret_h5"] = g["TDD_CLSPRC"].shift(-HORIZON_DAYS) / df["TDD_CLSPRC"] - 1

    df["target_score_h5"] = (
        0.7 * df["future_short_ratio_median_h5"] + 0.3 * df["future_balance_ratio_h5"]
    )
    df["own_hist_q70"] = df.groupby("ISU_CD")["target_score_h5"].transform(
        lambda x: x.shift(1).expanding(min_periods=MIN_HIST_DAYS).quantile(Q)
    )
    df["target"] = (
        (df["target_score_h5"] >= df["own_hist_q70"]) & (df["target_score_h5"] > 0)
    ).astype(int)
    return df


# ------------------------------------------------------------------
# 4. 시계열 기반 분할
# ------------------------------------------------------------------
def time_split(df):
    df = df.dropna(subset=FEATURE_COLS + ["target"]).copy()
    unique_dates = sorted(df["date"].unique())

    test_dates  = unique_dates[-TEST_SIZE_DAYS:]
    val_dates   = unique_dates[-(TEST_SIZE_DAYS + VAL_SIZE_DAYS):-TEST_SIZE_DAYS]
    train_dates = unique_dates[: -(TEST_SIZE_DAYS + VAL_SIZE_DAYS)]

    return (
        df[df["date"].isin(train_dates)],
        df[df["date"].isin(val_dates)],
        df[df["date"].isin(test_dates)],
    )


# ------------------------------------------------------------------
# 5-1. 테이블 / 선형 모델
# ------------------------------------------------------------------
def train_tabular_models(train, val, test):
    X_train, y_train = train[FEATURE_COLS], train["target"]
    X_val,   y_val   = val[FEATURE_COLS],   val["target"]
    X_test,  y_test  = test[FEATURE_COLS],  test["target"]

    n_neg      = (y_train == 0).sum()
    n_pos      = (y_train == 1).sum()
    pos_weight = n_neg / max(n_pos, 1)
    class_w    = {0: 1.0, 1: pos_weight}

    models, results = {}, {}
    scaler_std = StandardScaler().fit(X_train)
    Xtr_s = scaler_std.transform(X_train)
    Xvl_s = scaler_std.transform(X_val)
    Xte_s = scaler_std.transform(X_test)

    # [1] Linear Regression
    linreg = LinearRegression()
    linreg.fit(Xtr_s, y_train)
    proba_linreg = linreg.predict(Xte_s).clip(0, 1)
    models["Linear Regression"] = (linreg, scaler_std)
    results["Linear Regression"] = {
        "AUC": roc_auc_score(y_test, proba_linreg),
        "F1":  f1_score(y_test, proba_linreg >= 0.5, zero_division=0),
        "best_thresh": 0.5,
    }

    # [2] Logistic Regression
    lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight="balanced")
    lr.fit(Xtr_s, y_train)
    proba_lr_val  = lr.predict_proba(Xvl_s)[:, 1]
    proba_lr_test = lr.predict_proba(Xte_s)[:, 1]
    thresh_lr = find_best_threshold(y_val, proba_lr_val)
    models["Logistic Regression"] = (lr, scaler_std)
    results["Logistic Regression"] = {
        "AUC": roc_auc_score(y_test, proba_lr_test),
        "F1":  f1_score(y_test, proba_lr_test >= thresh_lr, zero_division=0),
        "best_thresh": thresh_lr,
    }

    # [3] LightGBM
    gbm = lgb.LGBMClassifier(
        random_state=RANDOM_STATE, verbose=-1,
        n_estimators=1000, learning_rate=0.02, num_leaves=127,
        min_child_samples=20, class_weight=class_w,
        subsample=0.8, colsample_bytree=0.8,
    )
    gbm.fit(X_train, y_train, eval_set=[(X_val, y_val)], eval_metric="auc",
            callbacks=[lgb.early_stopping(80, verbose=False)])
    proba_gbm_val  = gbm.predict_proba(X_val)[:, 1]
    proba_gbm_test = gbm.predict_proba(X_test)[:, 1]
    thresh_gbm = find_best_threshold(y_val, proba_gbm_val)
    models["LightGBM"] = (gbm, None)
    results["LightGBM"] = {
        "AUC": roc_auc_score(y_test, proba_gbm_test),
        "F1":  f1_score(y_test, proba_gbm_test >= thresh_gbm, zero_division=0),
        "best_thresh": thresh_gbm,
    }

    # [4] XGBoost
    xgbm = xgb.XGBClassifier(
        n_estimators=1000, learning_rate=0.02, max_depth=7,
        scale_pos_weight=pos_weight, random_state=RANDOM_STATE,
        eval_metric="auc", early_stopping_rounds=80, verbosity=0,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
    )
    xgbm.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    proba_xgb_val  = xgbm.predict_proba(X_val)[:, 1]
    proba_xgb_test = xgbm.predict_proba(X_test)[:, 1]
    thresh_xgb = find_best_threshold(y_val, proba_xgb_val)
    models["XGBoost"] = (xgbm, None)
    results["XGBoost"] = {
        "AUC": roc_auc_score(y_test, proba_xgb_test),
        "F1":  f1_score(y_test, proba_xgb_test >= thresh_xgb, zero_division=0),
        "best_thresh": thresh_xgb,
    }

    # [5] CatBoost
    catm = cb.CatBoostClassifier(
        iterations=1000, learning_rate=0.02, depth=7,
        scale_pos_weight=pos_weight, random_state=RANDOM_STATE,
        verbose=0, l2_leaf_reg=5.0, min_data_in_leaf=20,
    )
    catm.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=80)
    proba_cat_val  = catm.predict_proba(X_val)[:, 1]
    proba_cat_test = catm.predict_proba(X_test)[:, 1]
    thresh_cat = find_best_threshold(y_val, proba_cat_val)
    models["CatBoost"] = (catm, None)
    results["CatBoost"] = {
        "AUC": roc_auc_score(y_test, proba_cat_test),
        "F1":  f1_score(y_test, proba_cat_test >= thresh_cat, zero_division=0),
        "best_thresh": thresh_cat,
    }

    return models, results


# ------------------------------------------------------------------
# 5-2. LSTM
# ------------------------------------------------------------------
class LSTMClassifier(nn.Module):
    def __init__(self, n_features, hidden_size=32, num_layers=1):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden_size, num_layers=num_layers, batch_first=True)
        self.fc   = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze(-1)


def make_sequences(df, dates_allowed):
    Xs, ys = [], []
    for isu, g in df.groupby("ISU_CD"):
        g = g.sort_values("date").reset_index(drop=True)
        feat   = g[FEATURE_COLS].to_numpy()
        target = g["target"].to_numpy()
        valid  = ~np.isnan(feat).any(axis=1) & ~np.isnan(target)
        for i in range(LOOKBACK - 1, len(g)):
            if g["date"].iloc[i] not in dates_allowed:
                continue
            window = slice(i - LOOKBACK + 1, i + 1)
            if not valid[window].all():
                continue
            Xs.append(feat[window])
            ys.append(target[i])
    return np.asarray(Xs, dtype=np.float32), np.asarray(ys, dtype=np.float32)


def train_lstm(df, train, val, test, epochs=10, batch_size=512):
    Xtr,   ytr   = make_sequences(df, set(train["date"]))
    Xval,  yval  = make_sequences(df, set(val["date"]))
    Xtest, ytest = make_sequences(df, set(test["date"]))

    n, t, f = Xtr.shape
    scaler  = StandardScaler().fit(Xtr.reshape(-1, f))
    Xtr     = scaler.transform(Xtr.reshape(-1, f)).reshape(n, t, f)
    Xval    = scaler.transform(Xval.reshape(-1, f)).reshape(Xval.shape[0], t, f)
    Xtest   = scaler.transform(Xtest.reshape(-1, f)).reshape(Xtest.shape[0], t, f)

    model  = LSTMClassifier(n_features=f).to(DEVICE)
    opt    = torch.optim.Adam(model.parameters(), lr=1e-3)
    lossfn = nn.BCEWithLogitsLoss()

    train_dl = DataLoader(
        TensorDataset(torch.tensor(Xtr), torch.tensor(ytr)),
        batch_size=batch_size, shuffle=True,
    )
    val_tensor = torch.tensor(Xval).to(DEVICE)

    best_auc, best_state, val_pred_best = -1, None, None
    for _ in range(epochs):
        model.train()
        for xb, yb in train_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            nn.BCEWithLogitsLoss()(model(xb), yb).backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            val_pred = torch.sigmoid(model(val_tensor)).cpu().numpy()
        auc = roc_auc_score(yval, val_pred)
        if auc > best_auc:
            best_auc       = auc
            val_pred_best  = val_pred
            best_state     = {k: v.clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        test_pred = torch.sigmoid(model(torch.tensor(Xtest).to(DEVICE))).cpu().numpy()

    thresh_lstm = find_best_threshold(yval, val_pred_best)
    return model, scaler, {
        "AUC": roc_auc_score(ytest, test_pred),
        "F1":  f1_score(ytest, test_pred >= thresh_lstm, zero_division=0),
        "best_thresh": thresh_lstm,
    }


# ------------------------------------------------------------------
# 6. 종목 조회 유틸 (LightGBM 기준)
# ------------------------------------------------------------------
def predict_for(model, df, isu_cd, date, threshold=0.5):
    isu_cd = str(isu_cd).zfill(6)
    date   = pd.to_datetime(date)
    row    = df[(df["ISU_CD"] == isu_cd) & (df["date"] == date)]
    if row.empty:
        raise ValueError(f"데이터 없음: ISU_CD={isu_cd}, date={date.date()}")
    if row[FEATURE_COLS].isna().any(axis=1).iloc[0]:
        raise ValueError(f"피처 결측으로 예측 불가: ISU_CD={isu_cd}, date={date.date()}")
    prob = model.predict_proba(row[FEATURE_COLS])[:, 1][0]
    return {
        "ISU_CD":                      isu_cd,
        "ISU_NM":                      row["ISU_NM"].iloc[0],
        "date":                        date.date(),
        "prob_short_concentration_5d": round(float(prob), 4),
        "signal":                      int(prob >= threshold),
        "short_ratio_today":           row["short_ratio"].iloc[0],
        "short_ratio_pct_vs_own_hist": round(float(row["short_ratio_pct_vs_own_hist"].iloc[0]), 3),
        "balance_ratio_today":         row["balance_ratio"].iloc[0],
    }


def predict_history(model, df, isu_cd, threshold=0.5):
    isu_cd = str(isu_cd).zfill(6)
    sub    = df[df["ISU_CD"] == isu_cd].dropna(subset=FEATURE_COLS).copy()
    sub["prob_short_concentration_5d"] = model.predict_proba(sub[FEATURE_COLS])[:, 1]
    sub["signal"] = (sub["prob_short_concentration_5d"] >= threshold).astype(int)
    return sub[["ISU_CD", "ISU_NM", "date", "prob_short_concentration_5d",
                "signal", "short_ratio", "balance_ratio", "target"]].reset_index(drop=True)


# ------------------------------------------------------------------
# main
# ------------------------------------------------------------------
if __name__ == "__main__":
    df = load_data()
    df = add_features(df)
    df = add_target(df)

    train, val, test = time_split(df)
    print(f"Train: {train['date'].nunique()}일 / Val: {val['date'].nunique()}일 / Test: {test['date'].nunique()}일")
    print(f"Train target 비율: {train['target'].mean():.3f} / Test target 비율: {test['target'].mean():.3f}\n")

    # 테이블 모델 학습
    models, results = train_tabular_models(train, val, test)

    # LSTM
    t0 = time.time()
    _, _, lstm_result = train_lstm(df, train, val, test, epochs=10)
    results["LSTM"] = lstm_result
    print(f"(LSTM 학습 시간: {time.time()-t0:.1f}s)\n")

    # 결과 출력
    LINEAR_MODELS    = {"Linear Regression", "Logistic Regression"}
    NONLINEAR_MODELS = {"LightGBM", "XGBoost", "CatBoost", "LSTM"}

    print(f"{'Model':>22s} | {'AUC':>6s} | {'F1':>6s} | {'Thresh':>6s} | Type")
    print("-" * 65)
    for name, m in results.items():
        mtype = "선형" if name in LINEAR_MODELS else "비선형"
        print(f"{name:>22s} | {m['AUC']:.4f} | {m['F1']:.4f} | {m['best_thresh']:.3f}  | {mtype}")

    lin_f1s = [results[k]["F1"] for k in LINEAR_MODELS if k in results]
    nl_f1s  = [results[k]["F1"] for k in NONLINEAR_MODELS if k in results]
    if lin_f1s and nl_f1s:
        gap = np.mean(nl_f1s) - np.mean(lin_f1s)
        print(f"\n비선형 평균 F1: {np.mean(nl_f1s):.4f} | 선형 평균 F1: {np.mean(lin_f1s):.4f} | 차이: {gap:+.4f}")

    # 모델 + 데이터 저장 (XAI 모듈에서 로드)
    gbm        = models["LightGBM"][0]
    gbm_thresh = results["LightGBM"]["best_thresh"]

    joblib.dump({"models": models, "results": results, "gbm_thresh": gbm_thresh}, "models.joblib")
    df.to_pickle("df_full.pkl")
    print("\n모델 저장 완료 → models.joblib / df_full.pkl")
    print("XAI 분석은 short_xai.py 를 실행하세요.")

Train: 193일 / Val: 20일 / Test: 20일
Train target 비율: 0.370 / Test target 비율: 0.421

(LSTM 학습 시간: 23.3s)

                 Model |    AUC |     F1 | Thresh | Type
-----------------------------------------------------------------
     Linear Regression | 0.8226 | 0.7267 | 0.500  | 선형
   Logistic Regression | 0.8247 | 0.7361 | 0.520  | 선형
              LightGBM | 0.8493 | 0.7639 | 0.410  | 비선형
               XGBoost | 0.8482 | 0.7637 | 0.430  | 비선형
              CatBoost | 0.8433 | 0.7602 | 0.450  | 비선형
                  LSTM | 0.8584 | 0.7636 | 0.310  | 비선형

비선형 평균 F1: 0.7628 | 선형 평균 F1: 0.7314 | 차이: +0.0314

모델 저장 완료 → models.joblib / df_full.pkl
XAI 분석은 short_xai.py 를 실행하세요.


## short_xai.py : XAI분석

In [3]:



RANDOM_STATE  = 42
TEST_SIZE_DAYS = 20
VAL_SIZE_DAYS  = 20

FEATURE_COLS = [
    "log_mktcap", "log_price", "is_large_cap",
    "ret_1d", "ret_5d", "ret_20d", "vol_5d", "vol_20d", "intraday_vol", "price_ma20_gap",
    "turnover", "log_trdval", "abnormal_vol", "trdval_ma5_over_ma20", "ret5_x_abnormal_vol",
    "short_ratio_ma5", "short_ratio_ma10", "short_ratio_lag1",
    "short_ratio_chg_1d", "short_ratio_std5",
    "balance_ratio_chg_5d", "balance_ratio_ma5", "balance_ratio_lag1", "short_qty_chg5",
    "short_ratio_pct_vs_own_hist", "balance_ratio_pct_vs_own_hist",
]

"""
short_xai.py  —  XAI 분석 전용 모듈 (SHAP + DiCE)
사전 조건: short_model.py 를 먼저 실행해 models.joblib / df_full.pkl 생성

실행:
  python short_xai.py

출력 파일:
  shap_summary.png          — SHAP 전체 변수 중요도
  shap_dep_<feature>.png    — 상위 3개 변수 Dependence Plot
  shap_importance.csv       — 변수별 평균 |SHAP| 수치 테이블 (논문용)
  shap_force_<idx>.html     — 개별 샘플 Force Plot (사례 분석용)
  dice_results.csv          — DiCE 반사실적 시나리오 (논문용)
"""

import joblib
import numpy as np
import pandas as pd
import shap
import matplotlib
matplotlib.use("Agg")          # 서버 환경 GUI 없을 때 안전하게 사용
import matplotlib.pyplot as plt
import dice_ml
from dice_ml import Dice


# ------------------------------------------------------------------
# 저장 파일 로드
# ------------------------------------------------------------------
print("모델 및 데이터 로드 중...")
bundle = joblib.load("models.joblib")
models     = bundle["models"]
results    = bundle["results"]
gbm_thresh = bundle["gbm_thresh"]

df = pd.read_pickle("df_full.pkl")

# 시계열 분할 재현 (short_model.py 와 동일 로직)
unique_dates = sorted(df["date"].unique())
test_dates   = unique_dates[-TEST_SIZE_DAYS:]
val_dates    = unique_dates[-(TEST_SIZE_DAYS + VAL_SIZE_DAYS):-TEST_SIZE_DAYS]
train_dates  = unique_dates[: -(TEST_SIZE_DAYS + VAL_SIZE_DAYS)]

df_clean = df.dropna(subset=FEATURE_COLS + ["target"]).copy()
train = df_clean[df_clean["date"].isin(train_dates)]
val   = df_clean[df_clean["date"].isin(val_dates)]
test  = df_clean[df_clean["date"].isin(test_dates)]

gbm = models["LightGBM"][0]

# XAI용 샘플 (test set 최대 500개)
X_sample = test[FEATURE_COLS].sample(min(500, len(test)), random_state=RANDOM_STATE)
print(f"XAI 샘플 크기: {len(X_sample)}개\n")


# ==================================================================
# ① SHAP
# ==================================================================

def run_shap(model, X_sample: pd.DataFrame):
    """
    SHAP TreeExplainer 실행 후
      - shap_summary.png
      - shap_dep_<top3>.png
      - shap_importance.csv
      - shap_force_0.html (첫 번째 샘플 Force Plot)
    를 생성한다.
    """
    print("=" * 50)
    print("[SHAP] 분석 시작")
    print("=" * 50)

    explainer   = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_sample)

    # LightGBM 버전에 따라 shap_values 형태가 다름 → 양성 클래스 추출
    if isinstance(shap_values, list):
        sv = shap_values[1]                          # list[class0, class1]
    elif shap_values.ndim == 3:
        sv = shap_values[:, :, 1]                   # (n, feat, class)
    else:
        sv = shap_values                             # (n, feat)

    # ── (a) Summary Plot ──────────────────────────────────────────
    shap.summary_plot(sv, X_sample, show=False)
    plt.tight_layout()
    plt.savefig("shap_summary.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("  → shap_summary.png 저장 완료")

    # ── (b) 변수 중요도 테이블 (논문 본문용 수치) ─────────────────
    mean_abs_shap = np.abs(sv).mean(axis=0)
    importance_df = pd.DataFrame({
        "feature":       FEATURE_COLS,
        "mean_abs_shap": mean_abs_shap,
    }).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
    importance_df.to_csv("shap_importance.csv", index=False)
    print("  → shap_importance.csv 저장 완료")
    print("\n  [상위 10개 변수]")
    print(importance_df.head(10).to_string(index=False))

    # ── (c) Dependence Plot — 상위 3개 변수 ──────────────────────
    top3 = importance_df["feature"].head(3).tolist()
    for feat in top3:
        shap.dependence_plot(feat, sv, X_sample, show=False)
        plt.tight_layout()
        fname = f"shap_dep_{feat}.png"
        plt.savefig(fname, dpi=150, bbox_inches="tight")
        plt.close()
        print(f"  → {fname} 저장 완료")

    # ── (d) Force Plot — 첫 번째 샘플 (개별 종목 사례 분석) ──────
    # HTML 형태로 저장 → 논문엔 캡처본 삽입
    expected_val = (
        explainer.expected_value[1]
        if isinstance(explainer.expected_value, (list, np.ndarray))
        else explainer.expected_value
    )
    force_plot = shap.force_plot(
        expected_val,
        sv[0],
        X_sample.iloc[0],
        show=False,
    )
    shap.save_html("shap_force_0.html", force_plot)
    print("  → shap_force_0.html 저장 완료")

    return explainer, sv, importance_df


# ==================================================================
# ② DiCE
# ==================================================================

def run_dice(
    model,
    train: pd.DataFrame,
    test: pd.DataFrame,
    n_cf: int = 3,
    n_samples: int = 5,
):
    """
    DiCE 반사실적 설명 생성

    Parameters
    ----------
    n_cf       : 샘플당 생성할 반사실 수
    n_samples  : 분석할 '신호 없음' 샘플 수

    출력
    ----
    dice_results.csv — 원본 입력값 + 반사실 시나리오 (변경된 피처만 표시)
    """
    print("\n" + "=" * 50)
    print("[DiCE] 반사실적 설명 시작")
    print("=" * 50)

    # DiCE 데이터 래퍼
    dice_data = dice_ml.Data(
        dataframe=train[FEATURE_COLS + ["target"]].dropna(),
        continuous_features=FEATURE_COLS,
        outcome_name="target",
    )

    # DiCE 모델 래퍼 (sklearn 호환 predict_proba 사용)
    dice_model = dice_ml.Model(model=model, backend="sklearn")

    exp = Dice(dice_data, dice_model, method="random")

    # '신호 없음(0)'으로 예측된 샘플 추출
    proba_test = model.predict_proba(test[FEATURE_COLS])[:, 1]
    no_signal_mask = proba_test < gbm_thresh
    no_signal_df   = test[no_signal_mask][FEATURE_COLS].head(n_samples)

    if no_signal_df.empty:
        print("  경고: 신호 없음 샘플이 없습니다. 임계값을 낮춰주세요.")
        return None

    all_cfs = []
    for idx in range(len(no_signal_df)):
        query = no_signal_df.iloc[[idx]]
        try:
            cf_obj = exp.generate_counterfactuals(
                query,
                total_CFs=n_cf,
                desired_class="opposite",   # 0 → 1 로 전환하는 조건 탐색
            )
            cf_df = cf_obj.cf_examples_list[0].final_cfs_df
            if cf_df is not None:
                cf_df.insert(0, "sample_idx", idx)
                all_cfs.append(cf_df)
        except Exception as e:
            print(f"  샘플 {idx} DiCE 실패: {e}")

    if not all_cfs:
        print("  DiCE 결과 없음.")
        return None

    result_df = pd.concat(all_cfs, ignore_index=True)
    result_df.to_csv("dice_results.csv", index=False)
    print(f"  → dice_results.csv 저장 완료 ({len(result_df)}행)")

    # 콘솔에 변경된 피처만 요약 출력
    print("\n  [DiCE 요약 — 원본 대비 변경된 피처]")
    original = no_signal_df.reset_index(drop=True)
    for sample_idx in result_df["sample_idx"].unique():
        orig_row  = original.iloc[sample_idx]
        cf_rows   = result_df[result_df["sample_idx"] == sample_idx].drop(
            columns=["sample_idx", "target"], errors="ignore"
        )
        print(f"\n  ▶ 샘플 {sample_idx} (공매도 압력 신호 없음 → 있음 전환 조건)")
        for cf_i, (_, cf_row) in enumerate(cf_rows.iterrows()):
            changed = {
                feat: f"{orig_row[feat]:.4f} → {cf_row[feat]:.4f}"
                for feat in FEATURE_COLS
                if abs(orig_row[feat] - cf_row[feat]) > 1e-6
            }
            print(f"    반사실 {cf_i+1}: {changed}")

    return result_df


# ------------------------------------------------------------------
# main
# ------------------------------------------------------------------
if __name__ == "__main__":
    # SHAP
    explainer, sv, importance_df = run_shap(gbm, X_sample)

    # DiCE
    dice_result = run_dice(gbm, train, test, n_cf=3, n_samples=5)

    print("\n" + "=" * 50)
    print("XAI 분석 완료. 생성된 파일 목록:")
    print("  shap_summary.png")
    print("  shap_dep_<feature>.png  (상위 3개)")
    print("  shap_importance.csv")
    print("  shap_force_0.html")
    print("  dice_results.csv")
    print("=" * 50)

모델 및 데이터 로드 중...
XAI 샘플 크기: 500개

[SHAP] 분석 시작


LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray


  → shap_summary.png 저장 완료
  → shap_importance.csv 저장 완료

  [상위 10개 변수]
                      feature  mean_abs_shap
  short_ratio_pct_vs_own_hist       2.353703
              short_ratio_ma5       1.214202
             short_ratio_lag1       0.702664
           short_ratio_chg_1d       0.507931
                      vol_20d       0.186564
             short_ratio_std5       0.163384
balance_ratio_pct_vs_own_hist       0.138864
                   log_mktcap       0.126162
             short_ratio_ma10       0.113431
                       ret_5d       0.097058
  → shap_dep_short_ratio_pct_vs_own_hist.png 저장 완료
  → shap_dep_short_ratio_ma5.png 저장 완료
  → shap_dep_short_ratio_lag1.png 저장 완료
  → shap_force_0.html 저장 완료

[DiCE] 반사실적 설명 시작


100%|█████████████████████████████████████████████| 1/1 [00:02<00:00,  2.18s/it]

  → dice_results.csv 저장 완료 (15행)

  [DiCE 요약 — 원본 대비 변경된 피처]

  ▶ 샘플 0 (공매도 압력 신호 없음 → 있음 전환 조건)
    반사실 1: {'abnormal_vol': '2.0678 → 7.8000', 'short_ratio_pct_vs_own_hist': '0.1379 → 0.7000'}
    반사실 2: {'price_ma20_gap': '-0.0925 → 0.1000', 'short_ratio_pct_vs_own_hist': '0.1379 → 0.9000'}
    반사실 3: {'price_ma20_gap': '-0.0925 → 1.7000', 'short_ratio_pct_vs_own_hist': '0.1379 → 0.9000'}

  ▶ 샘플 1 (공매도 압력 신호 없음 → 있음 전환 조건)
    반사실 1: {'balance_ratio_lag1': '0.0000 → 7.3000', 'short_ratio_pct_vs_own_hist': '0.2910 → 1.0000'}
    반사실 2: {'turnover': '0.0054 → 4.6000', 'short_ratio_pct_vs_own_hist': '0.2910 → 0.7000'}
    반사실 3: {'balance_ratio_lag1': '0.0000 → 1.5000', 'short_ratio_pct_vs_own_hist': '0.2910 → 1.0000'}

  ▶ 샘플 2 (공매도 압력 신호 없음 → 있음 전환 조건)
    반사실 1: {'short_ratio_ma5': '0.5340 → 55.3000', 'short_ratio_chg_1d': '0.5800 → 92.1000'}
    반사실 2: {'ret_1d': '0.0098 → 0.0000', 'short_ratio_pct_vs_own_hist': '0.3143 → 0.9000'}
    반사실 3: {'ret_1d': '0.0098 → 0.4000', 'short_rat

In [6]:
import joblib
bundle = joblib.load("models.joblib")
print(bundle.keys())
print(list(bundle["models"].keys()))

dict_keys(['models', 'results', 'gbm_thresh'])
['Linear Regression', 'Logistic Regression', 'LightGBM', 'XGBoost', 'CatBoost']


In [7]:
import subprocess
subprocess.run(["pkill", "-f", "streamlit"])

  Stopping...


CompletedProcess(args=['pkill', '-f', 'streamlit'], returncode=0)